In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource']


# DB-Specific

In [3]:
from lib import allmusic
mio   = allmusic.MusicDBIO(verbose=False)
webio = allmusic.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant AllMusic DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AllMusic


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
knownArtists       = mio.data.getSummaryNameData()
searchArtists      = mio.data.getSearchArtistData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Local Artists:             {0}".format(len(localArtists.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

AllMusic Search Results
   Global Master Search:      761013
   Global Master Search Data: 0
   Local Artists:             219420
   Errors:                    470
   Search Artists:            1803604
   Known Summary IDs:         718650


# Search For New Artists

In [ ]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [ ]:
useKnownData  = False
useMasterData = True

if useKnownData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].notna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useMasterData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].isna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  50696
#   Artist Names To Get:  42803
#   Artist Names To Get:  33587
#   Artist Names To Get:  21888
#   Artist Names To Get:  10630

## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "7:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue
    if searchedForErrors.get(artistName) is not None:
        continue

    try:
        response = webio.getArtistSearchData(artistName=artistName)
    except:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(60)
        continue
        
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

## Save Results

In [ ]:
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        searchData = {}
        for searchTerm,searchTermData in mad.items():
            if isinstance(searchTermData,list):
                for item in searchTermData:
                    if isinstance(item,dict):
                        artistID = item['id'][2:] if isinstance(item.get('id'),str) else None
                        if artistID is not None:
                            searchData[artistID] = {k: v for k,v in item.items() if k in ['name','ref']}
        df = DataFrame(searchData).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [ ]:
mad = masterArtistsData.get()
df = getResponseDataFrame(mad)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = allmusic.MusicDBIO(local=False).data.getSearchArtistData()
    prevNewArtists = len(searchDF[~searchDF.index.isin(knownArtists.index)])
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF, df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("  ==> {0} Old Artists".format(searchDF[searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} New Artists".format(searchDF[~searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} Delta New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])-prevNewArtists))
    print("Saving Data")
    allmusic.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)
    
#Found 1628828 Unique Results
#Found 1639245 Unique Results
#Found 1664572 Unique Results
#Found 1674908 Unique Results

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [6]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [21]:
useKnown = False

artistNames       = searchArtists
localArtistsDict  = localArtists.get()
artistNamesToGet  = artistNames[~artistNames.index.isin(localArtistsDict.keys())]
if useKnown is True:
    pdbio = PanDBIO()
    pdbio.setData()
    matchedIDs = pdbio.getNotNaDBIDs(db)[db]
    artistNamesToGet = artistNamesToGet[artistNamesToGet.index.isin(matchedIDs)]
    del pdbio

print("{0} Search Results".format(db))
print("   Available Names:      {0}".format(len(artistNames)))
print("   Known Artist Names:   {0}".format(len(localArtistsDict)))
print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))

#   Artist Names To Get:  1561609
#   Artist Names To Get:  1526909

AllMusic Search Results
   Available Names:      1803604
   Known Artist Names:   303120
   Artist Names To Get:  1500484


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "6:50am")
tt = TermTime("today", "7:00pm")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["name"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting AllMusic ArtistIDs] Start    ==> Time Is 2022-05-03 06:51:40
========================= termTime(day=today,time=7:00pm) =========================
   ====> Terminate Time Set To 2022-05-03 19:00:00 <====
   ====> Will Terminate Process 12 Hours and 8 Minutes From Now
Getting Edward "Bunny" Lee (0001355742)                   True
Getting Glen Adams & the Bunny Lee All Stars (0000540513) True
Getting Bunny "Striker" Lee All Stars (0003295756)        True
Getting Bunny "Striker" Lee & the Roots of Reggae (0003295755)True
Getting Lee Bone (0000393626)                             True
Getting Lee Bean (0001843041)                             True
Getting Lee Bunnell (0001774277)                          True
Getting Lee Bunell (0002404598)                           True
Getting Drokz, Tails & Akira (0000804331)                 True
Getting MC Drokz (0001555906)                             True
Getting Count Megadrive & Mc Drokz (0001838309)           True
Getting Triaxis (000

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting John Anthony (0002405514)                         True
Getting John Anthony (0003870761)                         True
Getting John Anthony Alvarado (0000780379)                True
Getting John Anthony Ilvento (0001196818)                 True
Getting John Anthony Maltese (0001469736)                 True
Getting John Anthony Brisbin (0001786538)                 True
Getting John Anthony Rizzo (0001812550)                   True
Getting John Maltese (0001878462)                         True
Getting John Anthony Cruz (0002296775)                    True
Getting John Anthony Dawkins (0002513246)                 True
Getting Michael Sterling (0002194586)                     True
Getting Michael Sterling (0002078248)                     True
Getting Michael Sterling (0002464015)                     True
Getting Alozade (0000013828)                              True
Getting Michael Sterling Eaton (0002504264)               True
Getting Michael "Alozade" Sterling (0001075222)        

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Rev. D.C. Rice (0001197002)                       True
Getting Rising Kovenant (0003696333)                      True
Getting Cf98 (0003254667)                                 True
Getting Covenent (0000134242)                             True
Getting Covenants (0004194465)                            True
Getting Tribal Tech (0002911528)                          True
Getting Tribal & Tech (0002796516)                        True
Getting Sator Musicae Ensemble (0002194578)               True
Getting Sator Marte (0001527098)                          True
Getting Sator Malus (0003623870)                          True
Getting Minoru Yagyu (0001843035)                         True
Getting Haruna Yagyu (0001864778)                         True
Getting Makoto Yagyu (0002117330)                         True
Getting Sumimaro Yagyu (0002255100)                       True
Getting Masayoshi Yagyu (0003844019)                      True
Getting Toshihiko Yagyu (0004019044)                   

...

Getting Steven D. Smith (0001047093)                      True
Getting Steven K. Smith (0001555874)                      True
Getting Steven D. Smith (0001651623)                      True
Getting Steven G. Smith (0001856610)                      True
Getting Steven Maxwell Smith (0002151015)                 True
Getting Steven L. Smith (0002497406)                      True
Getting Steven Ross Smith (0002561116)                    True
Getting Steven Lee Smith (0002857900)                     True
Getting Steven W. Smith (0003009494)                      True
Getting Prince the King (0004134897)                      True
Getting Prince & The Seraphim (0000302651)                True
Getting Manny & the Revolution (0003134021)               True
Getting Toys for the Revolution (0000011657)              True
Getting Rise of the Revolution (0001486835)               True
Getting Mysteries of the Revolution (0001574017)          True
Getting Pride of the Revolution (0002006875)           

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Wang Xiaoming (0002882184)                        True
Getting Wang Zheng (0002183824)                           True
Getting Wang Jie (0003000413)                             True
Getting Wang Mei (0003458326)                             True
Getting Wang Breidahl (0003715186)                        True
Getting Ulla Wang (0002199512)                            True
Getting Wang Yi Ping (0003559133)                         True
Getting Riot (0000866533)                                 True
Getting Riot (0001023290)                                 True
Getting R.I.O.T. (0001223892)                             True
Getting Riot (0001314144)                                 True
Getting The Riot (0001335601)                             True
Getting Riot (0001542568)                                 True
Getting Riot (0001600698)                                 True
Getting Riot (0002056891)                                 True
Getting Riot (0002079201)                              

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Kasperi Viitanen (0002235562)                     True
Getting Kasperi Puranen (0003334357)                      True
Getting Kasperi Kallio (0003893137)                       True
Getting Kasperi Makinen (0004174938)                      True
Getting Kasperi Pitkanen (0004199216)                     True
Getting Paul Buckmaster / Harvey (0002275454)             True
Getting 20-20 (0000434107)                                True
Getting 20/20 (0001549796)                                True
Getting 20/20 (0001603013)                                True
Getting 20/20 (0001895997)                                True
Getting 20-20 (0002079394)                                True
Getting 2020 (0002648227)                                 True
Getting 20/20 Hindsight (0002027007)                      True
Getting 20/20 Faith (0003443899)                          True
Getting The 20/20 Project (0003512327)                    True
Getting MD 20-20 (0000182916)                          

In [66]:
searchArtists.loc['0000545523']['ref']

'https://www.allmusic.com/artist/gucci-mane-mn0000545523'

In [ ]:
mio.data.getRawFilename(mio.getModVal(artistID), artistID).str

In [43]:
retval = RawDBData().getRawData(mio.data.getRawFilename(23, '0000545523'))
retval.show()

Artist Data Class
-------------------------
Artist:  Gucci Mane
Meta:    Gucci Mane Songs, Albums, Reviews, Bio & More | AllMusic
         https://www.allmusic.com/artist/mn0000545523
Info:    <fileutils.info.FileInfo object at 0x7fa57f8a7430>
         2022-04-20 20:43:25.508787
         2022-04-21 10:08:03.090564
URL:     https://www.allmusic.com/artist/gucci-mane-mn0000545523
ID:      0000545523
Profile: {'general': {'moods': [<base.mdbrawbase.RawLinkData object at 0x7fa5980dbf10>, <base.mdbrawbase.RawLinkData object at 0x7fa5980dbf40>, <base.mdbrawbase.RawLinkData object at 0x7fa5980dbfa0>, <base.mdbrawbase.RawLinkData object at 0x7fa5980dbf70>, <base.mdbrawbase.RawLinkData object at 0x7fa5980e0070>, <base.mdbrawbase.RawLinkData object at 0x7fa5980e0040>, <base.mdbrawbase.RawLinkData object at 0x7fa5980e0130>, <base.mdbrawbase.RawLinkData object at 0x7fa5980e0190>, <base.mdbrawbase.RawLinkData object at 0x7fa5980e01f0>, <base.mdbrawbase.RawLinkData object at 0x7fa5980e0250>, <base.m

In [24]:
from ioutils import HTMLIO, FileIO
io  = FileIO()
hio = HTMLIO()
bsdata = hio.get(io.get(mio.data.getRawFilename(23, '0000545523')))
scriptData = bsdata.find("script", {"type": "application/ld+json"})

In [33]:

links = 

[<li class="tab overview" data-sections="moods,themes">
 <a href="/artist/gucci-mane-mn0000545523" title="Overview">
             Overview
             <span class="arrow">↓</span>
 </a>
 </li>,
 <li class="tab biography">
 <a href="/artist/gucci-mane-mn0000545523/biography" title="Biography">
                 Biography
                 <span class="arrow">↓</span>
 </a>
 </li>,
 <li class="tab discography">
 <a href="/artist/gucci-mane-mn0000545523/discography" title="Discography">
                 Discography
                 <span class="arrow">↓</span>
 </a>
 </li>,
 <li class="tab songs">
 <a href="/artist/gucci-mane-mn0000545523/songs" title="Songs">
                 Songs
                 <span class="arrow">↓</span>
 </a>
 </li>,
 <li class="tab credits">
 <a href="/artist/gucci-mane-mn0000545523/credits" title="Credits">
                 Credits
                 <span class="arrow">↓</span>
 </a>
 </li>,
 <li class="tab awards">
 <a href="/artist/gucci-mane-mn0000545523/awards

In [ ]:
""" Raw Data Storage Class """

__all__ = ["RawDBData"]

from base import RawDataBase
from .musicdbid import MusicDBID
from hashlib import md5
from strUtils import fixName

class RawDBData(RawDataBase):
    def __init__(self, debug=False):
        super().__init__()
        self.aid = MusicDBID()
        self.getArtistData      = self.getRawData
        self.getSongData        = self.getRawData
        self.getCreditData      = self.getRawData
        self.getCompositionData = self.getRawData
        
        
    ##############################################################################################################################
    ## Parse Data
    ##############################################################################################################################
    def getRawData(self, inputdata):
        self.getPickledHTMLData(inputdata)
        self.assertData()
        data                = {}
        data["artist"]      = self.getName()
        data["meta"]        = self.getMeta()
        data["url"]         = self.getURL()
        data["ID"]          = self.getID(data["url"])
        data["pages"]       = self.getPages()
        data["profile"]     = self.getProfile()
        data["media"]       = self.getMedia(data["url"])
        data["mediaCounts"] = self.makeRawMediaCountsData(counts={mediaType: len(mediaTypeData) for mediaType,mediaTypeData in data["media"].media.items()})
        data["info"]        = self.getInfo()
        return self.makeRawData(**data)
    

    ##############################################################################################################################
    ## Artist Name
    ##############################################################################################################################
    def getName(self):
        artistBios = self.bsdata.findAll("div", {"class": "artist-bio-container"})
        if len(artistBios) > 0:
            for div in artistBios:
                h1 = div.find("h1", {"class": "artist-name"})
                if h1 is not None:
                    artistName = h1.text.strip()
                    if len(artistName) > 0:
                        artist = fixName(artistName)
                        anc = self.makeRawNameData(name=artist, err=None)
                    else:
                        artist = "?"
                        anc = self.makeRawNameData(name=artist, err="Fix")
                else:
                    anc = self.makeRawNameData(err="NoH1")
        else:       
            anc = self.makeRawNameData(err=True)
            return anc
        
        return anc
    
    

    ##############################################################################################################################
    ## Meta Information
    ##############################################################################################################################
    def getMeta(self):
        metatitle = self.bsdata.find("meta", {"property": "og:title"})
        metaurl   = self.bsdata.find("meta", {"property": "og:url"})

        title = None
        if metatitle is not None:
            title = metatitle.attrs['content']

        url = None
        if metatitle is not None:
            url = metaurl.attrs['content']

        amc = self.makeRawMetaData(title=title, url=url)
        return amc
        

    ##############################################################################################################################
    ## Artist URL
    ##############################################################################################################################
    def getURL(self):
    <meta property="og:url" content="https://www.allmusic.com/artist/gucci-mane-mn0000545523">

        <meta name="spotim-ads" content="disable-all"/>
    
    <link rel="canonical" href="https://www.allmusic.com/artist/gucci-mane-mn0000545523">        
        
        result2 = self.bsdata.find("link", {"hreflang": "en"})
        if result1 and not result2:
            result = result1
        elif result2 and not result1:
            result = result2
        elif result1 and result2:
            result = result1
        else:        
            auc = self.makeRawURLData(err=True)
            return auc

        if result:
            url = result.attrs["href"]
            url = url.replace("https://www.allmusic.com", "")
            if url.find("/artist/") == -1:
                url = None
                auc = self.makeRawURLData(url=url, err="NoArtist")
            else:
                auc = self.makeRawURLData(url=url)
        else:
            auc = self.makeRawURLData(err="NoLink")

        return auc

    

    ##############################################################################################################################
    ## Artist ID
    ##############################################################################################################################
    def getID(self, suburl):
        discID = self.aid.getArtistID(suburl.url)
        aic = self.makeRawIDData(ID=discID)
        return aic


    
    ##############################################################################################################################
    ## Artist Pages
    ##############################################################################################################################
    def getPages(self):
        apc   = self.makeRawPageData(ppp=1, tot=1, redo=False, more=False)
        return apc
    
    

    ##############################################################################################################################
    ## Artist Variations
    ##############################################################################################################################
    def getProfile(self):
        generalData = None
        genreData   = None
        tagsData    = None
        extraData   = None

        content     = self.bsdata.find("meta", {"name": "title"})
        contentAttr = content.attrs if content is not None else None
        searchTerm  = contentAttr.get("content") if contentAttr is not None else None
        searchData  = [self.makeRawTextData(searchTerm)] if searchTerm is not None else None
        
        tabsul      = self.bsdata.find("ul", {"class": "tabs"})
        #print('tabsul',tabsul)
        refs        = tabsul.findAll("a") if tabsul is not None else None
        #print('refs',refs)
        tabLinks    = [self.makeRawLinkData(ref) for ref in refs] if refs is not None else None
        #print('tabLinks',tabLinks)
        #print('tabLinks',[x.get() for x in tabLinks])
        tabKeys = []
        if isinstance(tabLinks, list):
            for i,tabLink in enumerate(tabLinks):
                keyTitle = tabLink.title
                keyText  = tabLink.text
                if keyTitle is not None:
                    tabKeys.append(keyTitle)
                    continue
                if keyText is not None:
                    key = keyText.replace("\n", "").split()[0]
                    tabKeys.append(key)
                    continue
                tabKeys.append("Tab {0}".format(i))
        else:
            tabKeys = None
            
        tabsData    = dict(zip(tabKeys, tabLinks)) if (isinstance(tabKeys, list) and all(tabKeys)) else None
        #print('tabsData', tabsData)

        if searchData is not None:
            if extraData is None:
                extraData = {}
            extraData["Search"] = searchData
        if tabsData is not None:
            if extraData is None:
                extraData = {}
            extraData["Tabs"] = tabsData
        #print('extraData',extraData)


        basicInfo = self.bsdata.find("section", {"class": "basic-info"})
        if basicInfo is not None:
            for div in basicInfo.findAll("div"):
                attrs = div.attrs.get('class')
                if isinstance(attrs, list) and len(attrs) == 1:
                    attrKey = attrs[0]
                    if attrKey == "genre":
                        refs = div.findAll("a")
                        val  = [self.makeRawTextData(div)] if len(refs) == 0 else [self.makeRawLinkData(ref) for ref in refs]
                        genreData = val
                    elif attrKey == "styles":
                        refs = div.findAll("a")
                        val  = [self.makeRawTextData(div)] if len(refs) == 0 else [self.makeRawLinkData(ref) for ref in refs]
                        tagsData = val
                    else:
                        if generalData is None:
                            generalData = {}
                        refs = div.findAll("a")
                        val  = [self.makeRawTextData(div)] if len(refs) == 0 else [self.makeRawLinkData(ref) for ref in refs]
                        generalData[attrKey] = val

        apc = self.makeRawProfileData(general=generalData, tags=tagsData, genres=genreData, extra=extraData)
        return apc

    
    
    ##############################################################################################################################
    ## Artist Media
    ##############################################################################################################################
    def getMediaAlbum(self, td):
        for span in td.findAll("span"):
            attrs = span.attrs
            if attrs.get("class"):
                if 'format' in attrs["class"]:
                    albumformat = span.text
                    albumformat = albumformat.replace("(", "")
                    albumformat = albumformat.replace(")", "")
                    amac.format = albumformat
                    continue
            span.replaceWith("")

        ref = td.find("a")
        if ref:
            url    = ref.attrs['href']
            album  = ref.text
            retval = self.makeRawURLInfoData(url=url, name=album)
        else:
            retval = self.makeRawURLInfoData(url=None, name=None, err="NoText")

        return retval
    
    
    #def getMediaCredits(self):
    def getMediaSongs(self):
        mediaType = "Songs"
        media  = {}
        tables = self.bsdata.findAll("table")
        for table in tables:
            trs = table.findAll("tr")

            header  = trs[0]
            ths     = header.findAll("th")
            headers = [x.text.strip() for x in ths]
            if len(headers) == 0:
                continue
            for j,tr in enumerate(trs[1:]):
                tds  = tr.findAll("td")
                vals = [td.text.strip() for td in tds]

                tdTitle   = tr.find("td", {"class": "title-composer"})
                divTitle  = tdTitle.find("div", {"class": "title"}) if tdTitle is not None else None
                compTitle = tdTitle.find("div", {"class": "composer"}) if tdTitle is not None else None

                songTitle = divTitle.text if divTitle is not None else None
                songTitle = songTitle.strip() if songTitle is not None else None
                songURL   = divTitle.find('a') if divTitle is not None else None
                songURL   = self.makeRawLinkData(songURL) if songURL is not None else None
                
                if songTitle is None:
                    continue

                songArtists = compTitle.findAll("a") if compTitle is not None else None
                if songArtists is not None:
                    if len(songArtists) == 0:
                        songArtists = [self.makeRawTextData(compTitle.text)]
                    else:
                        songArtists = [self.makeRawLinkData(ref) for ref in songArtists]
                        
                m = md5()
                m.update(str(j).encode('utf-8'))
                if songTitle is not None:
                    m.update(songTitle.encode('utf-8'))
                code = str(int(m.hexdigest(), 16) % int(1e5))

                amdc = self.makeRawMediaReleaseData(album=songTitle, url=songURL, aclass=None, aformat=None, artist=songArtists, code=code, year=None)
                if media.get(mediaType) is None:
                    media[mediaType] = []
                media[mediaType].append(amdc)
                
        return media
        
        
    def getMediaCompositions(self):
        mediaType = "Composition"
        media = {}
        tables = self.bsdata.findAll("table")
        for table in tables:
            trs = table.findAll("tr")

            header  = trs[0]
            ths     = header.findAll("th")
            headers = [x.text.strip() for x in ths]
            if len(headers) == 0:
                continue
            for tr in trs[1:]:
                tds  = tr.findAll("td")
                vals = [td.text.strip() for td in tds]
                if len(vals) == len(headers):
                    albumData = dict(zip(headers,vals))

                    url   = None
                    year  = albumData.get('Year')
                    album = albumData.get('Title')
                    
                    if album is None:
                        continue

                    mediaType = "Composition"
                    for k,v in albumData.items():
                        if k.find("Genre") != -1:
                            mediaType = v

                    m = md5()
                    if year is not None:
                        m.update(year.encode('utf-8'))
                    if album is not None:
                        m.update(album.encode('utf-8'))
                    code = str(int(m.hexdigest(), 16) % int(1e5))

                    amdc = self.makeRawMediaReleaseData(album=album, url=url, aclass=None, aformat=None, artist=None, code=code, year=year)
                    if media.get(mediaType) is None:
                        media[mediaType] = []
                    media[mediaType].append(amdc)
              
        return media
            
                
    def getMedia(self, urlData):
        url  = urlData.url
        
        #print(url,'\t',end="")

        mediaData = {}
        if url is None:
            mediaType = "Unknown"
        else:
            mediaType = "Albums"
            if url.find("/credits") != -1:
                mediaType = "Credits"
            if url.find("/songs") != -1:
                mediaType = "Songs"
            if url.find("/compositions") != -1:
                mediaType = "Compositions"

        name = mediaType
        #print(mediaType)
                
        tables = self.bsdata.findAll("table")
        for table in tables:
            trs = table.findAll("tr")

            header  = trs[0]
            ths     = header.findAll("th")
            headers = [x.text.strip() for x in ths]
            if len(headers) == 0:
                continue

            for tr in trs[1:]:
                tds = tr.findAll("td")
                
                ## Name
                key = "Name"
                try:
                    if len(headers[1]) == 0:
                        idx = 1
                        mediaType = tds[idx].text.strip()
                        #print("From Text:",mediaType)
                        if len(mediaType) == 0:
                            mediaType = name
                            #print("From Name H:",mediaType)
                    else:
                        mediaType = name
                        #print("From Name:",mediaType)
                except:
                    #print("Error getting key: {0} from AllMusic media".format(key))
                    break

                ## Year
                key  = "Year"
                try:
                    idx  = headers.index(key)
                    year = tds[idx].text.strip()
                except:
                    #print("Error getting key: {0} from AllMusic media".format(key))
                    continue

                ## Title
                key   = "Album"
                try:
                    idx   = headers.index(key)
                    ref   = tds[idx].findAll("a")
                except:
                    #print("Error getting key: {0} from AllMusic media".format(key))
                    continue
                    
                try:
                    refdata = ref[0]
                    url     = refdata.attrs['href']
                    album   = refdata.text.strip()
                    
                    data = url.split("/")[-1]
                    pos  = data.rfind("-")
                    discIDurl = data[(pos+3):]       
                    discID = discIDurl.split("/")[0]

                    try:
                        int(discID)
                        code = discID
                    except:
                        code = None
                except:
                    url  = None
                    code = None
                    continue

                amdc = self.makeRawMediaReleaseData(album=album, url=url, aclass=None, aformat=None, artist=None, code=code, year=year)
                if mediaData.get(mediaType) is None:
                    mediaData[mediaType] = []
                mediaData[mediaType].append(amdc)

                
        compMedia = self.getMediaCompositions()
        for mediaType,mediaTypeData in compMedia.items():
            if mediaData.get(mediaType) is None:
                mediaData[mediaType] = []
            mediaData[mediaType] += mediaTypeData
            

        songMedia = self.getMediaSongs()
        for mediaType,mediaTypeData in songMedia.items():
            if mediaData.get(mediaType) is None:
                mediaData[mediaType] = []
            mediaData[mediaType] += mediaTypeData
                
        return self.makeRawMediaData(media=mediaData)